# Aula 13 - Detecção de Anomalias: Gerador de Figuras para Slides

Este notebook gera todas as figuras Plotly usadas nos slides da Aula 13.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.utils import PlotlyJSONEncoder
from pathlib import Path
import json

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DRACULA = {
    'bg': '#282a36',
    'fg': '#f8f8f2',
    'cyan': '#8be9fd',
    'green': '#50fa7b',
    'pink': '#ff79c6',
    'red': '#ff5555',
    'yellow': '#f1fa8c',
    'purple': '#bd93f9',
    'muted': '#6272a4',
}

def apply_dracula(fig, title=None):
    fig.update_layout(
        title=title,
        paper_bgcolor=DRACULA['bg'],
        plot_bgcolor=DRACULA['bg'],
        font=dict(color=DRACULA['fg']),
        margin=dict(l=60, r=30, t=60, b=55),
        showlegend=True,
        legend=dict(orientation='h', y=-0.25),
    )
    fig.update_xaxes(gridcolor='#44475a', zerolinecolor=DRACULA['muted'])
    fig.update_yaxes(gridcolor='#44475a', zerolinecolor=DRACULA['muted'])
    return fig

def clean_none(obj):
    if isinstance(obj, dict):
        return {k: clean_none(v) for k, v in obj.items() if v is not None}
    if isinstance(obj, list):
        return [clean_none(v) for v in obj]
    return obj

def slide_export_path():
    cwd = Path.cwd().resolve()
    if cwd.name == 'notebooks' and cwd.parent.name == 'ciencia-dados':
        return cwd.parent / 'images' / 'plotly' / 'aula13_anomaly_detection_figures.js'
    return cwd / 'ciencia-dados' / 'images' / 'plotly' / 'aula13_anomaly_detection_figures.js'

def write_plotly_js(figures, output_path):
    lines = []
    for name, fig in figures.items():
        payload = clean_none(fig.to_dict())
        traces = json.dumps(payload['data'], ensure_ascii=False, cls=PlotlyJSONEncoder)
        layout = json.dumps(payload['layout'], ensure_ascii=False, cls=PlotlyJSONEncoder)
        lines.append(f"const {name}_TRACES = {traces};\n")
        lines.append(f"const {name}_LAYOUT = {layout};\n")
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text('\n'.join(lines), encoding='utf-8')

## 1. Dataset sintético: clientes

In [2]:
n_normal = 300

normais = pd.DataFrame({
    "gasto_mensal": np.random.normal(500, 120, n_normal),
    "numero_compras": np.random.normal(12, 3, n_normal)
})

anomalias_reais = pd.DataFrame({
    "gasto_mensal": [1200, 1400, 100, 1600, 80],
    "numero_compras": [2, 3, 40, 1, 45]
})

dados = pd.concat([normais, anomalias_reais], ignore_index=True)
dados["anomalia_real"] = [0] * n_normal + [1] * len(anomalias_reais)

print(f"Total: {len(dados)} pontos | Normais: {n_normal} | Anomalias: {len(anomalias_reais)}")

Total: 305 pontos | Normais: 300 | Anomalias: 5


## 2. Slide: Motivação — exemplos do mundo real

In [3]:
# Simulação de transações financeiras: a maioria normal, algumas suspeitas
np.random.seed(42)
n_tx = 200
transacoes = pd.DataFrame({
    "valor": np.random.exponential(150, n_tx),
    "hora": np.random.uniform(8, 22, n_tx),
})
transacoes_suspeitas = pd.DataFrame({
    "valor": [5000, 8000, 3500, 6200],
    "hora": [2, 3.5, 1, 4],
})
transacoes["tipo"] = "Normal"
transacoes_suspeitas["tipo"] = "Suspeita"
tx_all = pd.concat([transacoes, transacoes_suspeitas], ignore_index=True)

fig_motivacao = go.Figure()
fig_motivacao.add_trace(go.Scatter(
    x=tx_all.loc[tx_all["tipo"] == "Normal", "hora"],
    y=tx_all.loc[tx_all["tipo"] == "Normal", "valor"],
    mode='markers', name='Normal',
    marker=dict(color=DRACULA['cyan'], size=7, opacity=0.6),
))
fig_motivacao.add_trace(go.Scatter(
    x=tx_all.loc[tx_all["tipo"] == "Suspeita", "hora"],
    y=tx_all.loc[tx_all["tipo"] == "Suspeita", "valor"],
    mode='markers', name='Suspeita',
    marker=dict(color=DRACULA['red'], size=14, symbol='x', line=dict(width=3)),
))
fig_motivacao.update_xaxes(title="Hora do dia")
fig_motivacao.update_yaxes(title="Valor da transação (R$)")
fig_motivacao = apply_dracula(fig_motivacao, "Transações financeiras: quais parecem suspeitas?")
fig_motivacao.show()

## 3. Slide: Dataset de clientes — visão geral

In [4]:
fig_scatter_initial = go.Figure()
fig_scatter_initial.add_trace(go.Scatter(
    x=dados.loc[dados["anomalia_real"] == 0, "gasto_mensal"],
    y=dados.loc[dados["anomalia_real"] == 0, "numero_compras"],
    mode='markers', name='Normal',
    marker=dict(color=DRACULA['cyan'], size=8, opacity=0.6),
    hovertemplate='Gasto: R$%{x:.0f}<br>Compras: %{y:.0f}<extra></extra>',
))
fig_scatter_initial.add_trace(go.Scatter(
    x=dados.loc[dados["anomalia_real"] == 1, "gasto_mensal"],
    y=dados.loc[dados["anomalia_real"] == 1, "numero_compras"],
    mode='markers', name='Anomalia inserida',
    marker=dict(color=DRACULA['red'], size=16, symbol='x', line=dict(width=3)),
    hovertemplate='Gasto: R$%{x:.0f}<br>Compras: %{y:.0f}<extra></extra>',
))
fig_scatter_initial.update_xaxes(title="Gasto mensal (R$)")
fig_scatter_initial.update_yaxes(title="Número de compras")
fig_scatter_initial = apply_dracula(fig_scatter_initial, "Clientes: comportamento normal e anômalo")
fig_scatter_initial.show()

## 4. Slide: Z-score — distribuição e limites

In [5]:
x = dados["gasto_mensal"]
z = (x - x.mean()) / x.std()
anomalias_z = abs(z) > 3

fig_zscore = go.Figure()

# Histograma
fig_zscore.add_trace(go.Histogram(
    x=x, name='Distribuição',
    marker=dict(color=DRACULA['cyan'], opacity=0.7),
    nbinsx=30,
))

# Linhas de limite
media = x.mean()
std = x.std()
for mult, color, label in [(-3, DRACULA['red'], '-3σ'), (3, DRACULA['red'], '+3σ'), (0, DRACULA['yellow'], 'média')]:
    fig_zscore.add_vline(x=media + mult*std, line_color=color, line_dash='dash', line_width=2,
                         annotation_text=label, annotation_font_color=color)

# Pontos anômalos destacados
anom_vals = x[anomalias_z]
fig_zscore.add_trace(go.Scatter(
    x=anom_vals, y=[30]*len(anom_vals), mode='markers', name='Anomalia (|z|>3)',
    marker=dict(color=DRACULA['red'], size=12, symbol='x', line=dict(width=3)),
))

fig_zscore.update_xaxes(title="Gasto mensal (R$)")
fig_zscore.update_yaxes(title="Frequência")
fig_zscore = apply_dracula(fig_zscore, f"Z-score: {anomalias_z.sum()} anomalias detectadas (|z| > 3)")
fig_zscore.show()

## 5. Slide: IQR — boxplot com outliers

In [6]:
fig_boxplot = go.Figure()

fig_boxplot.add_trace(go.Box(
    y=dados["gasto_mensal"], name="Gasto mensal",
    marker_color=DRACULA['cyan'],
    boxmean=True,
    hoverinfo='y',
))
fig_boxplot.add_trace(go.Box(
    y=dados["numero_compras"], name="Número de compras",
    marker_color=DRACULA['pink'],
    boxmean=True,
    hoverinfo='y',
))

fig_boxplot.update_yaxes(title="Valor")
fig_boxplot = apply_dracula(fig_boxplot, "Boxplot: outliers identificados pelo IQR")
fig_boxplot.show()

In [7]:
# IQR detection
def detectar_iqr(serie):
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    lim_inf = q1 - 1.5 * iqr
    lim_sup = q3 + 1.5 * iqr
    return (serie < lim_inf) | (serie > lim_sup)

outlier_gasto = detectar_iqr(dados["gasto_mensal"])
outlier_compras = detectar_iqr(dados["numero_compras"])
dados["anomalia_iqr"] = outlier_gasto | outlier_compras
n_iqr = dados["anomalia_iqr"].sum()

fig_iqr_scatter = go.Figure()
fig_iqr_scatter.add_trace(go.Scatter(
    x=dados.loc[~dados["anomalia_iqr"], "gasto_mensal"],
    y=dados.loc[~dados["anomalia_iqr"], "numero_compras"],
    mode='markers', name='Normal',
    marker=dict(color=DRACULA['cyan'], size=8, opacity=0.6),
))
fig_iqr_scatter.add_trace(go.Scatter(
    x=dados.loc[dados["anomalia_iqr"], "gasto_mensal"],
    y=dados.loc[dados["anomalia_iqr"], "numero_compras"],
    mode='markers', name='Detectado pelo IQR',
    marker=dict(color=DRACULA['red'], size=16, symbol='x', line=dict(width=3)),
))
fig_iqr_scatter.update_xaxes(title="Gasto mensal (R$)")
fig_iqr_scatter.update_yaxes(title="Número de compras")
fig_iqr_scatter = apply_dracula(fig_iqr_scatter, f"IQR: {n_iqr} anomalias detectadas")
fig_iqr_scatter.show()

## 6. Slide: Limitação univariado — combinação estranha

In [8]:
# Cria um ponto que é normal em cada variável isolada mas anômalo na combinação
np.random.seed(42)
n = 200
x1 = np.random.normal(500, 100, n)
x2 = np.random.normal(12, 3, n)

# Ponto "escondido": gasto médio-Alto mas compras muito altas — não é outlier em nenhuma variável sozinha
ponto_escondido = pd.DataFrame({"gasto": [520], "compras": [30]})

fig_multivariado = go.Figure()
fig_multivariado.add_trace(go.Scatter(
    x=x1, y=x2, mode='markers', name='Normal',
    marker=dict(color=DRACULA['cyan'], size=8, opacity=0.5),
))
fig_multivariado.add_trace(go.Scatter(
    x=ponto_escondido["gasto"], y=ponto_escondido["compras"],
    mode='markers', name='Anômalo na combinação',
    marker=dict(color=DRACULA['yellow'], size=18, symbol='star', line=dict(width=2, color=DRACULA['red'])),
))

# Linhas de referência: médias
fig_multivariado.add_vline(x=np.mean(x1), line_color=DRACULA['muted'], line_dash='dot',
                           annotation_text='média gasto')
fig_multivariado.add_hline(y=np.mean(x2), line_color=DRACULA['muted'], line_dash='dot',
                           annotation_text='média compras')

fig_multivariado.update_xaxes(title="Gasto mensal (R$)")
fig_multivariado.update_yaxes(title="Número de compras")
fig_multivariado = apply_dracula(fig_multivariado, "Anomalia multivariada: normal em cada variável, estranho na combinação")
fig_multivariado.show()

## 7. Slide: DBSCAN como detector de anomalias

In [9]:
X = dados[["gasto_mensal", "numero_compras"]].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

modelo_dbscan = DBSCAN(eps=0.8, min_samples=10)
labels_dbscan = modelo_dbscan.fit_predict(X_scaled)
dados["label_dbscan"] = labels_dbscan
dados["anomalia_dbscan"] = labels_dbscan == -1
n_dbscan = dados["anomalia_dbscan"].sum()

fig_dbscan = go.Figure()

# Pontos normais agrupados por cluster
unique_labels = set(labels_dbscan)
cluster_colors = [DRACULA['cyan'], DRACULA['pink'], DRACULA['green'], DRACULA['yellow'], DRACULA['purple']]
for i, label in enumerate(sorted([l for l in unique_labels if l >= 0])):
    mask = labels_dbscan == label
    fig_dbscan.add_trace(go.Scatter(
        x=X[mask, 0], y=X[mask, 1], mode='markers',
        name=f'Cluster {label}',
        marker=dict(color=cluster_colors[i % len(cluster_colors)], size=8, opacity=0.6),
    ))

# Ruído
noise_mask = labels_dbscan == -1
fig_dbscan.add_trace(go.Scatter(
    x=X[noise_mask, 0], y=X[noise_mask, 1], mode='markers',
    name=f'Ruído (anomalia)',
    marker=dict(color=DRACULA['red'], size=16, symbol='x', line=dict(width=3)),
))

fig_dbscan.update_xaxes(title="Gasto mensal (escalado)")
fig_dbscan.update_yaxes(title="Número de compras (escalado)")
fig_dbscan = apply_dracula(fig_dbscan, f"DBSCAN: {n_dbscan} pontos marcados como ruído (label=-1)")
fig_dbscan.show()

## 8. Slide: Isolation Forest

In [10]:
modelo_if = IsolationForest(contamination=0.05, random_state=RANDOM_STATE)
pred_if = modelo_if.fit_predict(X_scaled)
dados["anomalia_if"] = pred_if == -1
n_if = dados["anomalia_if"].sum()

fig_isolation = go.Figure()
fig_isolation.add_trace(go.Scatter(
    x=dados.loc[~dados["anomalia_if"], "gasto_mensal"],
    y=dados.loc[~dados["anomalia_if"], "numero_compras"],
    mode='markers', name='Normal',
    marker=dict(color=DRACULA['cyan'], size=8, opacity=0.6),
))
fig_isolation.add_trace(go.Scatter(
    x=dados.loc[dados["anomalia_if"], "gasto_mensal"],
    y=dados.loc[dados["anomalia_if"], "numero_compras"],
    mode='markers', name='Isolation Forest',
    marker=dict(color=DRACULA['red'], size=16, symbol='x', line=dict(width=3)),
))
fig_isolation.update_xaxes(title="Gasto mensal (R$)")
fig_isolation.update_yaxes(title="Número de compras")
fig_isolation = apply_dracula(fig_isolation, f"Isolation Forest: {n_if} anomalias detectadas")
fig_isolation.show()

## 9. Slide: Local Outlier Factor

In [11]:
modelo_lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
pred_lof = modelo_lof.fit_predict(X_scaled)
dados["anomalia_lof"] = pred_lof == -1
n_lof = dados["anomalia_lof"].sum()

fig_lof = go.Figure()
fig_lof.add_trace(go.Scatter(
    x=dados.loc[~dados["anomalia_lof"], "gasto_mensal"],
    y=dados.loc[~dados["anomalia_lof"], "numero_compras"],
    mode='markers', name='Normal',
    marker=dict(color=DRACULA['cyan'], size=8, opacity=0.6),
))
fig_lof.add_trace(go.Scatter(
    x=dados.loc[dados["anomalia_lof"], "gasto_mensal"],
    y=dados.loc[dados["anomalia_lof"], "numero_compras"],
    mode='markers', name='LOF',
    marker=dict(color=DRACULA['red'], size=16, symbol='x', line=dict(width=3)),
))
fig_lof.update_xaxes(title="Gasto mensal (R$)")
fig_lof.update_yaxes(title="Número de compras")
fig_lof = apply_dracula(fig_lof, f"LOF: {n_lof} anomalias detectadas")
fig_lof.show()

## 10. Slide: Comparação visual lado a lado

In [12]:
# Bar chart comparando métodos
def recall(real, prevista):
    vp = ((real == 1) & (prevista == True)).sum()
    total = (real == 1).sum()
    return vp / total if total > 0 else 0

comparacao = pd.DataFrame({
    "metodo": ["IQR", "DBSCAN", "Isolation Forest", "LOF"],
    "detectadas": [n_iqr, n_dbscan, n_if, n_lof],
    "recall": [
        recall(dados["anomalia_real"], dados["anomalia_iqr"]),
        recall(dados["anomalia_real"], dados["anomalia_dbscan"]),
        recall(dados["anomalia_real"], dados["anomalia_if"]),
        recall(dados["anomalia_real"], dados["anomalia_lof"]),
    ]
})

fig_comparacao = go.Figure()
colors = [DRACULA['cyan'], DRACULA['green'], DRACULA['purple'], DRACULA['yellow']]
fig_comparacao.add_trace(go.Bar(
    x=comparacao["metodo"], y=comparacao["detectadas"],
    marker_color=colors,
    text=comparacao["detectadas"], textposition='outside',
    textfont=dict(color=DRACULA['fg'], size=14),
))
fig_comparacao.update_xaxes(title="Método")
fig_comparacao.update_yaxes(title="Anomalias detectadas")
fig_comparacao = apply_dracula(fig_comparacao, "Comparação: quantas anomalias cada método encontrou?")
fig_comparacao.show()

In [13]:
# Recall chart
fig_recall = go.Figure()
fig_recall.add_trace(go.Bar(
    x=comparacao["metodo"], y=comparacao["recall"],
    marker_color=colors,
    text=[f"{r:.0%}" for r in comparacao["recall"]], textposition='outside',
    textfont=dict(color=DRACULA['fg'], size=14),
))
fig_recall.update_xaxes(title="Método")
fig_recall.update_yaxes(title="Recall (anomalias reais capturadas)", range=[0, 1.1])
fig_recall = apply_dracula(fig_recall, "Recall: quantas anomalias reais cada método encontrou?")
fig_recall.show()

## 11. Slide: Falso positivo vs falso negativo — matriz de confusão visual

In [14]:
# Matriz de confusão para Isolation Forest
tp = ((dados["anomalia_real"] == 1) & (dados["anomalia_if"] == True)).sum()
fp = ((dados["anomalia_real"] == 0) & (dados["anomalia_if"] == True)).sum()
fn = ((dados["anomalia_real"] == 1) & (dados["anomalia_if"] == False)).sum()
tn = ((dados["anomalia_real"] == 0) & (dados["anomalia_if"] == False)).sum()

fig_confusion = go.Figure()
fig_confusion.add_trace(go.Heatmap(
    z=[[tn, fp], [fn, tp]],
    x=['Previsto Normal', 'Previsto Anômalo'],
    y=['Real Normal', 'Real Anômalo'],
    text=[[f'VN\n{tn}', f'FP\n{fp}'], [f'FN\n{fn}', f'VP\n{tp}']],
    texttemplate='%{text}', textfont=dict(size=16, color=DRACULA['fg']),
    colorscale=[[0, DRACULA['cyan']], [1, DRACULA['red']]],
    showscale=False,
))
fig_confusion = apply_dracula(fig_confusion, "Matriz de confusão: Isolation Forest")
fig_confusion.update_xaxes(side='top')
fig_confusion.show()

print(f"Verdadeiros positivos: {tp}")
print(f"Falsos positivos: {fp}")
print(f"Falsos negativos: {fn}")
print(f"Verdadeiros negativos: {tn}")

Verdadeiros positivos: 5
Falsos positivos: 11
Falsos negativos: 0
Verdadeiros negativos: 289


## 12. Exportar figuras para JS

In [15]:
figures = {
    'AULA13_MOTIVACAO': fig_motivacao,
    'AULA13_SCATTER_INITIAL': fig_scatter_initial,
    'AULA13_ZSCORE': fig_zscore,
    'AULA13_BOXPLOT': fig_boxplot,
    'AULA13_IQR_SCATTER': fig_iqr_scatter,
    'AULA13_MULTIVARIADO': fig_multivariado,
    'AULA13_DBSCAN': fig_dbscan,
    'AULA13_ISOLATION': fig_isolation,
    'AULA13_LOF': fig_lof,
    'AULA13_COMPARACAO': fig_comparacao,
    'AULA13_RECALL': fig_recall,
    'AULA13_CONFUSION': fig_confusion,
}

write_plotly_js(figures, slide_export_path())
print(f"Exported {len(figures)} figures to {slide_export_path()}")

Exported 12 figures to /home/openclaw/opencode/reveal-cpvbcdd-202501/ciencia-dados/images/plotly/aula13_anomaly_detection_figures.js
